# HSK Graded Sentences

Rank `cmn_sentences_clean.tsv` from easiest → hardest.

**Pipeline**
1. Load word-frequency list → rarity rank per word.
2. Load New HSK (2025) word lists → HSK level (1–7, 8 = beyond-HSK) per word.
3. Seed jieba dict with HSK + top-freq words for clean segmentation.
4. Clean sentences: drop English-letter rows, re-convert to simplified, dedupe, length band 2–30 chars.
5. Segment each sentence → per-word HSK level + freq rank → sentence features.
6. **Hybrid difficulty** = weighted composite (continuous) + `max_hsk` grade band.
7. Add pinyin (pypinyin), sort, save `cmn_sentences_graded.tsv`.

**Difficulty formula** (each feature min-max normalised across corpus):
```
difficulty = 0.45*max_hsk + 0.25*mean_log_freq + 0.15*n_chars + 0.15*frac_oov
```

In [1]:
# !python -m pip install pypinyin jieba opencc
import csv
import re
import math
from pathlib import Path
from collections import Counter

import jieba
import opencc
from pypinyin import pinyin, Style

DATA = Path("data")
HSK_DIR = Path("New HSK (2025)")
SENTENCES_FILE = DATA / "cmn_sentences_clean.tsv"
FREQ_FILE = DATA / "blog_lit_news_tech_weibo_freq.release_sorted.txt"
OUTPUT_FILE = DATA / "cmn_sentences_graded.tsv"

BEYOND_HSK = 8           # level assigned to words in no HSK list
MIN_CHARS, MAX_CHARS = 2, 30

HAN = re.compile(r"[\u4e00-\u9fff]")
LATIN = re.compile(r"[A-Za-z]")
TRAIL_DIGITS = re.compile(r"[0-9]+$")

t2s = opencc.OpenCC("t2s")   # safety: force simplified everywhere
print("imports ok")

imports ok


In [2]:
# --- 1. Word frequency list: word -> 1-based rank (lower = more common) ---
freq_rank = {}
freq_count = {}
with open(FREQ_FILE, encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 2:
            continue
        w, c = parts
        if w and w not in freq_rank:    # first occurrence = best (highest) rank
            freq_rank[w] = i
            freq_count[w] = int(c)

MAX_RANK = len(freq_rank) + 1           # fallback rank for unseen words
print(f"freq words loaded: {len(freq_rank):,}   fallback rank = {MAX_RANK:,}")

freq words loaded: 1,617,835   fallback rank = 1,617,836


In [3]:
# --- 2. New HSK (2025): word -> lowest level (1..7, 7-9 collapsed to 7) ---
HSK_FILES = {
    1: "HSK_Level_1_words.txt",
    2: "HSK_Level_2_words.txt",
    3: "HSK_Level_3_words.txt",
    4: "HSK_Level_4_words.txt",
    5: "HSK_Level_5_words.txt",
    6: "HSK_Level_6_words.txt",
    7: "HSK_Level_7-9_words.txt",
}

hsk_level = {}
for lvl in sorted(HSK_FILES):
    with open(HSK_DIR / HSK_FILES[lvl], encoding="utf-8") as f:
        for line in f:
            w = TRAIL_DIGITS.sub("", line.strip())   # strip sense markers: 本1 -> 本
            w = t2s.convert(w)
            if w and w not in hsk_level:             # keep lowest level seen
                hsk_level[w] = lvl

print("HSK words:", len(hsk_level))
print("per level:", dict(sorted(Counter(hsk_level.values()).items())))

HSK words: 10896
per level: {1: 300, 2: 197, 3: 491, 4: 990, 5: 1579, 6: 1777, 7: 5562}


In [4]:
# --- 3. Seed jieba so multi-char HSK / common words segment as one token ---
for w in hsk_level:
    if len(w) >= 2:
        jieba.add_word(w)

added = 0
for w, r in freq_rank.items():
    if len(w) >= 2 and r <= 100_000:
        jieba.add_word(w)
        added += 1
print(f"jieba seeded: {len(hsk_level)} HSK words + {added:,} top-freq words")

Building prefix dict from the default dictionary ...


Loading model from cache /tmp/claude-501/jieba.cache


Loading model cost 0.247 seconds.


Prefix dict has been built successfully.


jieba seeded: 10896 HSK words + 91,429 top-freq words


In [5]:
# --- 4. Clean: simplified, drop English-letter rows, dedupe, length band ---
rows = []          # (id, text, n_chars)
seen = set()
stats = {"total": 0, "english": 0, "length": 0, "dupe": 0}

with open(SENTENCES_FILE, encoding="utf-8") as f:
    for r in csv.reader(f, delimiter="\t"):
        if len(r) != 3:
            continue
        stats["total"] += 1
        sid, _, text = r
        text = t2s.convert(text.strip())
        if LATIN.search(text):
            stats["english"] += 1
            continue
        n_chars = len(HAN.findall(text))
        if not (MIN_CHARS <= n_chars <= MAX_CHARS):
            stats["length"] += 1
            continue
        if text in seen:
            stats["dupe"] += 1
            continue
        seen.add(text)
        rows.append((sid, text, n_chars))

print(f"loaded {stats['total']:,} -> kept {len(rows):,}")
print("dropped:", {k: v for k, v in stats.items() if k != 'total'})

loaded 78,269 -> kept 75,510
dropped: {'english': 1321, 'length': 907, 'dupe': 531}


In [6]:
# --- 5. Segment + per-sentence features ---
def features(text):
    words = [w for w in jieba.lcut(text) if HAN.search(w)]
    if not words:
        return None
    n = len(words)
    levels = [hsk_level.get(w, BEYOND_HSK) for w in words]
    ranks = [freq_rank.get(w, MAX_RANK) for w in words]
    return {
        "n_words": n,
        "max_hsk": max(levels),
        "mean_hsk": sum(levels) / n,
        "hsk_coverage": sum(l <= 7 for l in levels) / n,
        "frac_oov": sum(l == BEYOND_HSK for l in levels) / n,
        "rarest_rank": max(ranks),
        "mean_log_freq": sum(math.log10(r) for r in ranks) / n,
    }

recs = []
for sid, text, n_chars in rows:
    feat = features(text)
    if feat is None:
        continue
    feat.update(id=sid, text=text, n_chars=n_chars)
    recs.append(feat)

print(f"scored {len(recs):,} sentences")

scored 75,510 sentences


In [7]:
# --- 6. Hybrid difficulty: min-max normalise features, weighted composite ---
def make_norm(key):
    vals = [r[key] for r in recs]
    lo, hi = min(vals), max(vals)
    span = (hi - lo) or 1.0
    return lambda v: (v - lo) / span

n_hsk = make_norm("max_hsk")
n_freq = make_norm("mean_log_freq")
n_len = make_norm("n_chars")
n_oov = make_norm("frac_oov")

W = dict(hsk=0.45, freq=0.25, length=0.15, oov=0.15)
for r in recs:
    r["difficulty"] = round(
        W["hsk"]    * n_hsk(r["max_hsk"])
        + W["freq"]   * n_freq(r["mean_log_freq"])
        + W["length"] * n_len(r["n_chars"])
        + W["oov"]    * n_oov(r["frac_oov"]),
        5,
    )

# easiest -> hardest; tie-break by grade band then rarity
recs.sort(key=lambda r: (r["difficulty"], r["max_hsk"], r["rarest_rank"]))
print("difficulty range:", recs[0]["difficulty"], "->", recs[-1]["difficulty"])

difficulty range: 0.00536 -> 0.94643


In [8]:
# --- 7. Pinyin + write graded TSV ---
def to_pinyin(text):
    return " ".join(seg[0] for seg in pinyin(text, style=Style.TONE))

COLS = [
    "rank", "id", "sentence", "pinyin", "n_chars", "n_words",
    "max_hsk", "mean_hsk", "hsk_coverage", "rarest_rank",
    "mean_log_freq", "difficulty",
]
with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(COLS)
    for i, r in enumerate(recs, 1):
        w.writerow([
            i, r["id"], r["text"], to_pinyin(r["text"]),
            r["n_chars"], r["n_words"], r["max_hsk"],
            round(r["mean_hsk"], 2), round(r["hsk_coverage"], 2),
            r["rarest_rank"], round(r["mean_log_freq"], 3), r["difficulty"],
        ])
print(f"saved {len(recs):,} graded sentences -> {OUTPUT_FILE}")

saved 75,510 graded sentences -> data/cmn_sentences_graded.tsv


In [9]:
# --- 8. Sanity preview ---
def show(label, items):
    print(f"\n=== {label} ===")
    for r in items:
        print(f"d={r['difficulty']:.3f} hsk{r['max_hsk']}  {to_pinyin(r['text'])}")
        print(f"    {r['text']}")

show("EASIEST 8", recs[:8])
show("HARDEST 8", recs[-8:])
print("\nmax_hsk band counts:", dict(sorted(Counter(r['max_hsk'] for r in recs).items())))


=== EASIEST 8 ===
d=0.005 hsk1  shì wǒ de 。
    是我的。
d=0.007 hsk1  wǒ zài 。
    我在。
d=0.009 hsk1  shì wǒ 。
    是我。
d=0.014 hsk1  hǎo de 。
    好的。
d=0.014 hsk1  shì nǐ ！
    是你！
d=0.016 hsk1  nǐ shì wǒ de 。
    你是我的。
d=0.016 hsk1  duì le ！
    对了！
d=0.017 hsk1  zhè shì wǒ de 。
    这是我的。

=== HARDEST 8 ===
d=0.887 hsk8  “ zěn me huí shì ？” xiǎo bái tù wèn dào 。
    “怎么回事？”小白兔问道。
d=0.887 hsk8  wǒ huì zài shì yī cì ,  xiè xiè nín 。
    我会再试一次, 谢谢您。
d=0.887 hsk8  nà běn shū réng shòu bǎn quán bǎo hù 。
    那本书仍受版权保护。
d=0.895 hsk8  sān fáng èr tīng èr wèi ， jiàn zhù miàn jī shì yì bǎi sān shí liù píng mǐ ， dé fáng lǜ bǎi fēn zhī qī shí 。
    三房二厅二卫，建筑面积是一百三十六平米，得房率百分之七十。
d=0.903 hsk8  wǒ ài māo ， gèng ài gǒu 。 huàn jù huà shuō  :  wǒ ài gǒu ， shèng yú ài māo 。
    我爱猫，更爱狗。换句话说 : 我爱狗，胜于爱猫。
d=0.908 hsk8  qiān lǐ zhī xíng shǐ yú zú xià ， jiǔ céng zhī tái qǐ yú lěi tǔ 。
    千里之行始于足下，九层之台起于垒土。
d=0.926 hsk8  shí 、 èr shí 、 sān shí 、 sì shí 、 wǔ shí 、 liù shí 、 qī shí 、 bā shí 、 jiǔ shí 、 yì bǎi 。


In [10]:
# --- 9. HSK word coverage: how many graded sentences contain each word ---
COVERAGE_FILE = DATA / "hsk_word_coverage.tsv"

word_sents = Counter()
for r in recs:
    for w in {t for t in jieba.lcut(r["text"]) if HAN.search(t)}:
        if w in hsk_level:
            word_sents[w] += 1

cov = {w: word_sents.get(w, 0) for w in hsk_level}   # every HSK word, 0 if unseen

# per-word file, sorted by level then count desc
with open(COVERAGE_FILE, "w", encoding="utf-8", newline="") as f:
    wr = csv.writer(f, delimiter="\t")
    wr.writerow(["word", "hsk_level", "n_sentences"])
    for w, n in sorted(cov.items(), key=lambda x: (hsk_level[x[0]], -x[1])):
        wr.writerow([w, hsk_level[w], n])

# level-wise summary
tot, zero = Counter(), Counter()
for w, lvl in hsk_level.items():
    tot[lvl] += 1
    if cov[w] == 0:
        zero[lvl] += 1

print(f"{'HSK':>4} {'words':>6} {'covered':>8} {'no_sent':>8} {'%cov':>6}")
for lvl in sorted(tot):
    c = tot[lvl] - zero[lvl]
    band = "7-9" if lvl == 7 else str(lvl)
    print(f"{band:>4} {tot[lvl]:>6} {c:>8} {zero[lvl]:>8} {100*c/tot[lvl]:>5.1f}%")
T, Z = sum(tot.values()), sum(zero.values())
print(f"{'ALL':>4} {T:>6} {T-Z:>8} {Z:>8} {100*(T-Z)/T:>5.1f}%")
print(f"\nsaved per-word coverage -> {COVERAGE_FILE}")
print("\nzero-coverage samples per level:")
for lvl in sorted(tot):
    zs = [w for w in hsk_level if hsk_level[w] == lvl and cov[w] == 0][:12]
    print(f"  L{'7-9' if lvl==7 else lvl}: {' '.join(zs)}")

 HSK  words  covered  no_sent   %cov
   1    300      296        4  98.7%
   2    197      196        1  99.5%
   3    491      484        7  98.6%
   4    990      952       38  96.2%
   5   1579     1471      108  93.2%
   6   1777     1466      311  82.5%
 7-9   5562     3218     2344  57.9%
 ALL  10896     8083     2813  74.2%

saved per-word coverage -> data/hsk_word_coverage.tsv

zero-coverage samples per level:
  L1: 百 没（有） 面条儿 有（一）点儿
  L2: 有时（候）
  L3: 东北 东南 短裤 方便面 检票 开机 前年
  L4: 笔试 差（一）点儿 车位 大赛 动车 法 父女 干活儿 高价 红包 话剧 获奖
  L5: 报到 本领 闭幕式 餐饮 城区 乘务员 车主 充值 窗台 创业 初级 除夕
  L6: 岸 案例 百分点 百货 白领 拜年 保健 保修 北美洲 倍增 本土 避
  L7-9: 癌 哀求 暗地里 按理 按说 安置 凹 坝 疤 把柄 霸道 把关
